# CMPS 355 — Corrected Nested LASSO & Elastic Net Diagnostic (v1)

This notebook mounts your personal Google Drive (`the authorized Google Drive account`), verifies the canonical dataset SHA-256 (`8a83f20035...e91a74`), and executes `corrected_nested_lasso.py` directly into:
`CMPS355_Research/peer_review_revision_2026/03_followup_experiments/corrected_nested_lasso_20261122_v1/`

- **Original project files and Phase 1 evidence are never modified.**
- All outputs (inner tuning scores, full coefficients + intercepts, KKT diagnostics, Stage 22 metrics + 2,000-resample bootstrap CIs, Elastic Net diagnostic, `INTERPRETATION_REPORT.md`, `HANDOFF_FOR_CHATGPT_AUDIT.md`, `manifest.json`, `SHA256SUMS.txt`) are written incrementally to Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib, platform, numpy, pandas, scipy, sklearn

ROOT = Path("/content/drive/MyDrive/CMPS355_Research")
DATA_ZIP = ROOT / "data/validated/ibd_dataset_sample_aligned.zip"
RUN_DIR = ROOT / "peer_review_revision_2026/03_followup_experiments/corrected_nested_lasso_20261122_v1"
SCRIPT = RUN_DIR / "corrected_nested_lasso.py"

assert DATA_ZIP.is_file(), f"Canonical dataset not found: {DATA_ZIP}"
assert SCRIPT.is_file(), f"Execution script not found: {SCRIPT}"
zsha = hashlib.sha256(DATA_ZIP.read_bytes()).hexdigest()
assert zsha == "8a83f20035fcac5ae5d8bfe496e4d7491e47400b61ca1080b828c8ad54e91a74", f"SHA mismatch: {zsha}"
print("Verified canonical ZIP SHA-256:", zsha)
print(f"Python {platform.python_version()} | NumPy {numpy.__version__} | SciPy {scipy.__version__} | pandas {pandas.__version__} | scikit-learn {sklearn.__version__}")

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(SCRIPT),
    "--data", str(DATA_ZIP),
    "--out", str(RUN_DIR),
    "--n-boot", "2000",
    "--en-max-iter", "20000",
]
print("Running:", " ".join(cmd))
log_path = RUN_DIR / "run.log"
with open(log_path, "w") as lf:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
        lf.write(line)
    rc = proc.wait()
assert rc == 0, f"Execution failed with exit code {rc}"
print("\nCOMPLETED SUCCESSFULLY. Log saved to:", log_path)

In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import pandas as pd

# Existing completed run — NO model fitting or experiments
out = Path(
    "/content/drive/MyDrive/CMPS355_Research/"
    "peer_review_revision_2026/03_followup_experiments/"
    "corrected_nested_lasso_20261122_v1"
)

# Verify that all expensive experiments produced their outputs
required_counts = {
    "loco_outer_lasso_summary.csv": 4,
    "indep_lasso_summary.csv": 4,
    "elastic_net_numerical_diagnostic.csv": 8,
    "stage22_corrected_lasso_predictions_PRIVATE.csv": 689,
    "loco_inner_tuning_scores.csv": 48,
    "indep_cv_tuning_scores.csv": 48,
}

for filename, expected in required_counts.items():
    file = out / filename
    assert file.is_file(), f"Missing: {filename}"
    actual = len(pd.read_csv(file))
    assert actual == expected, (
        f"{filename}: expected {expected} rows, got {actual}"
    )
    print(f"VERIFIED: {filename} ({actual} rows)")

for filename in [
    "manifest.json",
    "INTERPRETATION_REPORT.md",
    "HANDOFF_FOR_CHATGPT_AUDIT.md",
]:
    assert (out / filename).is_file(), f"Missing: {filename}"
    print(f"VERIFIED: {filename}")

# Record exactly why the original run stopped
note = out / "POSTRUN_RECOVERY_NOTE.md"
note.write_text(
    "# Post-run recovery\n\n"
    f"Recovery UTC: {datetime.now(timezone.utc).isoformat()}\n\n"
    "All four experiment parts finished and their saved outputs "
    "passed the expected row-count checks.\n\n"
    "The original script failed exclusively during the final "
    "SHA256SUMS generation because Google Drive exposes native "
    "Google Docs as unreadable .gdoc placeholders in Colab.\n\n"
    "This recovery generated checksums for ordinary readable files "
    "while excluding Google Workspace virtual placeholders.\n\n"
    "No models were refitted; historical and experimental "
    "results were not changed.\n"
)

# Complete ONLY the previously failed checksum step
virtual_extensions = {
    ".gdoc", ".gsheet", ".gslides",
    ".gdraw", ".gform", ".gmap", ".gsite"
}

checksums = []
skipped = []

for path in sorted(out.rglob("*")):
    if not path.is_file() or path.name == "SHA256SUMS.txt":
        continue

    if path.suffix.lower() in virtual_extensions:
        skipped.append(path.relative_to(out).as_posix())
        continue

    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)

    relative = path.relative_to(out).as_posix()
    checksums.append(f"{digest.hexdigest()}  {relative}")

checksum_file = out / "SHA256SUMS.txt"
checksum_file.write_text("\n".join(checksums) + "\n")

assert checksum_file.exists() and checksum_file.stat().st_size > 0

print("\n" + "=" * 55)
print("RECOVERY COMPLETED SUCCESSFULLY")
print("No experiments rerun.")
print(f"Files checksummed: {len(checksums)}")
print(f"Google Workspace placeholders skipped: {len(skipped)}")
print(f"Checksums saved: {checksum_file}")
print("=" * 55)


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

print("=== 1. LOCO OUTER LASSO SUMMARY ===")
display(pd.read_csv(RUN_DIR / "loco_outer_lasso_summary.csv"))

print("=== 2. STAGE 22 CONTROLLED COHORT METRICS ===")
display(pd.read_csv(RUN_DIR / "stage22_corrected_lasso_cohort_metrics.csv"))

print("=== 3. SENSITIVITY COMPARISON VS HISTORICAL & OTHER 4 METHODS ===")
display(pd.read_csv(RUN_DIR / "comparison_vs_historical_and_four_methods_SENSITIVITY.csv"))

print("=== 4. INDEPENDENT-COHORT LASSO SUMMARY (TABLE VI CONTEXT) ===")
display(pd.read_csv(RUN_DIR / "indep_lasso_summary.csv"))

print("=== 5. TARGETED ELASTIC NET NUMERICAL DIAGNOSTIC ===")
display(pd.read_csv(RUN_DIR / "elastic_net_numerical_diagnostic.csv"))

display(Markdown((RUN_DIR / "INTERPRETATION_REPORT.md").read_text()))